# Clase 3 REBOUND — Sistema completo: Apophis, Tierra, Luna, Júpiter y Sol

**Curso:** Mecánica Celeste  
**Tema:** Dinámica del encuentro cercano 99942 Apophis – Tierra (2029)  
**Sistema de referencia primario:** Baricentro del Sistema Solar (SSB)  
**Integradores:** IAS15 (exacto adaptativo) y WHFast (simpléctico)  

---

## Objetivos

1. Construir una simulación de **5 cuerpos** (Sol, Júpiter, Tierra, Luna, Apophis) con condiciones iniciales extraídas del **JPL Horizons** mediante la interfaz REBOUND.
2. Elegir el **baricentro del Sistema Solar (SSB)** como sistema de referencia y justificar esta elección frente al marco heliocéntrico.
3. Analizar el encuentro cercano del **13 de abril de 2029** con herramientas de mecánica celeste: distancia mínima, velocidad de encuentro, parámetro de impacto y factor de deflexión gravitacional.
4. Estudiar el **efecto de Júpiter y la Luna** sobre la geometría del encuentro mediante simulaciones con y sin esos cuerpos.
5. Cambiar al **marco geocéntrico** para visualizar la trayectoria de Apophis alrededor de la Tierra durante el *flyby*.
6. Producir **animaciones con Celluloid** en ambos sistemas de referencia.
7. Realizar un **análisis de Monte Carlo** para evaluar la sensibilidad de la distancia de máxima aproximación a incertidumbres en las condiciones iniciales.


---

## 1. Contexto científico: ¿Quién es Apophis?

El asteroide **99942 Apophis** (anteriormente 2004 MN₄) fue descubierto el 19 de junio de 2004 por Roy Tucker, David Tholen y Fabrizio Bernardi en el Observatorio Nacional de Kitt Peak [1]. Tras unos días de observación, en diciembre de 2004 se calculó una probabilidad de impacto con la Tierra en 2029 del **2.7 %**, la mayor registrada hasta entonces para un objeto de ese tamaño [2].

### Datos físicos principales

| Parámetro | Valor |
|---|---|
| Designación oficial | (99942) Apophis |
| Tipo espectral | Sq (silicatos + olivino) |
| Diámetro efectivo | ~340 m |
| Masa estimada | ~2.7 × 10¹⁰ kg |
| Albedo geométrico | 0.30 +/- 0.06 |
| Período orbital | 323.7 días |
| Semieje mayor | 0.9227 UA |
| Excentricidad | 0.1914 |
| Inclinación | 3.34° |

### Encuentros relevantes con la Tierra

| Fecha | Distancia (UA) | Distancia (LD) | Notas |
|---|---|---|---|
| **13 abr 2029** | **~0.000254 UA** | **~0.098 LD** | **Máxima aproximación confirmada** |
| 13 abr 2036 | ~0.0061 UA | ~2.4 LD | Depende del "keyhole" de 2029 |
| 2068 | TBD | TBD | Monitoreo continuo |

> *LD = distancia Tierra-Luna = 384 400 km*

En 2029, Apophis pasará **más cerca que muchos satélites geoestacionarios** (35 786 km de altitud). Con magnitud visual ~3.3, será **visible a simple vista** desde Europa, África y Asia occidental [3].

### Importancia de la simulación numérica

La predicción exacta de la trayectoria requiere incluir:  
- El frenado por **presión de radiación solar** (efecto Yarkovsky) — pequeño pero crítico a largo plazo.  
- Las perturbaciones gravitacionales de **todos los planetas**, especialmente **Júpiter**.  
- La atracción de la **Luna** durante el *flyby*.  
- La **relatividad general** en el campo del Sol.  

En este cuaderno usamos REBOUND con integrador **IAS15** que es prácticamente exacto a nivel de máquina para encuentros cercanos [4].

---

**Referencias**  
[1] Tholen D. J. et al. (2004) *IAUC 8477*.  
[2] Giorgini J. D. et al. (2008) *Icarus* 193, 1-19.  
[3] Pravec P. et al. (2014) *Icarus* 233, 48-60.  
[4] Rein H. & Spiegel D. S. (2015) *MNRAS* 446, 1424-1437.


---

## 2. Sistema de referencia: Baricentro del Sistema Solar (SSB)

### 2.1 ¿Por qué usar el SSB?

| Marco | Origen | Ventajas | Limitaciones |
|---|---|---|---|
| **Heliocéntrico eclíptico J2000** | Centro del Sol | Intuitivo, clásico | El Sol orbita el SSB ~1-2 R_sol |
| **SSB eclíptico J2000** | Baricentro del S.S. | Marco inercial real, conserva momento | Menos intuitivo |

El **SSB** es el **marco inercial natural** del Sistema Solar: en él, el momento lineal total es cero y las leyes de Newton se cumplen sin términos de arrastre. Las efemérides de alta precisión del JPL (DE441) están expresadas en el SSB [5].

### 2.2 Transformación SSB → Geocéntrico

Para el análisis del *flyby* usaremos también el marco geocéntrico:

```
r_geo = r_SSB - r_Tierra,SSB
v_geo = v_SSB - v_Tierra,SSB
```

### 2.3 Jerarquía de efectos gravitacionales sobre Apophis

| Cuerpo | GM (UA³/día²) | r típica (UA) | a (UA/día²) |
|---|---|---|---|
| Sol | 2.959e-4 | 1.0 | 2.96e-4 |
| Júpiter | 2.825e-7 | 4.2 | 1.6e-8 |
| Tierra | 8.888e-10 | 0.003 (flyby) | 9.9e-5 |
| Luna | 1.093e-11 | 0.004 (flyby) | 6.8e-7 |

Durante el *flyby*, la **Tierra supera al Sol** como fuente dominante de perturbación.

---
[5] Park R.S. et al. (2021) *AJ* 161, 105 — DE441.


In [ ]:
# ── BLOQUE 0 – Instalación e importaciones ────────────────────────────────
!pip install rebound celluloid numpy matplotlib astropy -q

import rebound
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from celluloid import Camera
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

# Constantes físicas
AU       = 1.495978707e11   # m
G_SI     = 6.674e-11        # m^3/(kg*s^2)
M_sun_kg = 1.989e30         # kg
day      = 86400            # s
AU2km    = 1.496e8          # UA -> km
AU2kkm   = 1.496e5          # UA -> 10^3 km

print(f'REBOUND version : {rebound.__version__}')
print('Todas las dependencias cargadas correctamente.')



---

## 3. Construcción de la simulación: 5 cuerpos en el SSB

### 3.1 Extracción de condiciones iniciales (JPL Horizons)

REBOUND puede contactar directamente la API de Horizons del JPL para obtener posiciones y velocidades baricentricas de cualquier objeto del Sistema Solar.

Usaremos la **época 1 de abril de 2029** (dos semanas antes del encuentro).

### 3.2 Lista de cuerpos

| # | Nombre | ID Horizons | Masa |
|---|---|---|---|
| 0 | Sol | `sun` | M_sol |
| 1 | Júpiter | `5` | 9.548e-4 M_sol |
| 2 | Tierra | `399` | 3.003e-6 M_sol |
| 3 | Luna | `301` | 3.693e-8 M_sol |
| 4 | Apophis | `2099942` | ~1.9e-17 M_sol |

> La masa de Apophis es **despreciable** para la dinámica del sistema, pero se mantiene para trazar su trayectoria.


In [ ]:
# ── BLOQUE 1 – Configurar la simulacion (5 cuerpos, IAS15) ─────────────────

EPOCH = '2029-Apr-01'

def crear_simulacion(epoch=EPOCH, integrador='ias15'):
    sim = rebound.Simulation()
    sim.units = ('AU', 'yr', 'Msun')   # G = 4*pi^2
    sim.integrator = integrador
    if integrador == 'ias15':
        sim.ri_ias15.epsilon = 1e-9

    cuerpos = [
        ('Sun',     'sun',    None),
        ('Jupiter', '5',      None),
        ('Earth',   '399',    None),
        ('Moon',    '301',    None),
        ('Apophis', '2099942', 1.9e-17),
    ]

    for name, horizons_id, mass_override in cuerpos:
        p = rebound.Particle(simulation=sim, primary=None,
                             date=epoch, id=horizons_id)
        if mass_override is not None:
            p.m = mass_override
        p.hash = name
        sim.add(p)

    sim.move_to_com()
    print(f'Simulacion creada: {len(sim.particles)} particulas en el SSB')
    print(f'Fecha de inicio  : {epoch}')
    print(f'Integrador       : {integrador.upper()}')
    return sim

sim = crear_simulacion()

# Imprimir estado inicial
nombres = ['Sol', 'Jupiter', 'Tierra', 'Luna', 'Apophis']
print(f"{'Cuerpo':<10} {'x (UA)':>12} {'y (UA)':>12} {'z (UA)':>12} {'m (M_sol)':>14}")
print('-' * 65)
for i, (p, n) in enumerate(zip(sim.particles, nombres)):
    print(f'{n:<10} {p.x:>12.6f} {p.y:>12.6f} {p.z:>12.6f} {p.m:>14.4e}')



In [ ]:
# ── BLOQUE 2 – Distancia inicial Tierra-Apophis ────────────────────────────

I_SUN=0; I_JUP=1; I_EAR=2; I_MOO=3; I_APO=4

earth   = sim.particles['Earth']
apophis = sim.particles['Apophis']

dx = apophis.x - earth.x
dy = apophis.y - earth.y
dz = apophis.z - earth.z
d0_UA = (dx**2 + dy**2 + dz**2)**0.5
d0_km = d0_UA * 1.496e8
d0_LD = d0_km / 384400

print(f'Distancia inicial Tierra-Apophis:')
print(f'  {d0_UA:.6f}  UA')
print(f'  {d0_km:.0f}  km')
print(f'  {d0_LD:.3f}  LD (distancias lunares)')



---

## 4. Integración numérica y extracción de trayectorias

Integramos desde el **1 de abril de 2029** hasta el **30 de abril de 2029** (30 días), guardando el estado a intervalos de **1 hora**.

El integrador **IAS15** ajusta automáticamente el paso de tiempo para mantener el error relativo por debajo de epsilon = 10^-9, lo que es crucial durante el encuentro cuando las fuerzas cambian rápidamente.

**Tiempo de integración**: 30 días × 24 pasos/día = 720 pasos guardados.


In [ ]:
# ── BLOQUE 3 – Integracion principal (30 dias, paso de salida 1 h) ─────────

T_total_dias = 30
dt_output_yr = 1.0 / (365.25 * 24)   # 1 hora en anios
N_steps      = int(T_total_dias * 24)
N_part       = len(sim.particles)

pos  = np.zeros((N_steps, N_part, 3))
vel  = np.zeros((N_steps, N_part, 3))
t_yr = np.zeros(N_steps)

sim_run = sim.copy()
sim_run.move_to_com()

print('Integrando...')
for i in range(N_steps):
    sim_run.integrate(sim_run.t + dt_output_yr)
    t_yr[i] = sim_run.t
    for j, p in enumerate(sim_run.particles):
        pos[i, j] = [p.x, p.y, p.z]
        vel[i, j] = [p.vx, p.vy, p.vz]

t_dias = t_yr * 365.25
print(f'Integracion completada: {N_steps} pasos, {T_total_dias} dias')
print(f'Tiempo final (dias): {t_dias[-1]:.2f}')



In [ ]:
# ── BLOQUE 4 – Distancia Tierra-Apophis y deteccion del flyby ──────────────

dr_EA    = pos[:, I_APO, :] - pos[:, I_EAR, :]
d_EA_UA  = np.linalg.norm(dr_EA, axis=1)
d_EA_km  = d_EA_UA * 1.496e8
d_EA_LD  = d_EA_km / 384400

i_min      = np.argmin(d_EA_UA)
d_min_UA   = d_EA_UA[i_min]
d_min_km   = d_EA_km[i_min]
d_min_LD   = d_EA_LD[i_min]
t_min_dias = t_dias[i_min]

# Velocidad relativa
AU_yr_to_km_s  = 1.496e8 / (365.25 * 86400)
dv_EA_raw      = vel[:, I_APO, :] - vel[:, I_EAR, :]
v_rel_km_s     = np.linalg.norm(dv_EA_raw, axis=1) * AU_yr_to_km_s
v_flyby        = v_rel_km_s[i_min]

print('=' * 55)
print('  RESULTADOS DEL FLYBY  --  SSB, IAS15, 5 cuerpos')
print('=' * 55)
print(f'  Dia del flyby (desde 1-abr-2029): {t_min_dias:.4f} dias')
print(f'  Distancia minima  : {d_min_UA:.6f} UA')
print(f'               km  : {d_min_km:.0f} km')
print(f'               LD  : {d_min_LD:.4f} LD')
print(f'  Velocidad relativa: {v_flyby:.3f} km/s')
print('=' * 55)
print()
print('Comparacion con datos JPL:')
print('  JPL horizons: ~38 000 km, ~7.4 km/s')
print(f'  Este modelo : {d_min_km:.0f} km, {v_flyby:.1f} km/s')



In [ ]:
# ── BLOQUE 5 – Figura 1: Distancia y velocidad Tierra-Apophis vs tiempo ─────

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

ax1 = axes[0]
ax1.plot(t_dias, d_EA_LD, color='#2196F3', lw=2, label='Distancia Tierra-Apophis')
ax1.axhline(1.0, color='#E91E63', ls='--', lw=1.5, label='1 LD (distancia lunar)')
ax1.axhline(0.1, color='#FF9800', ls='--', lw=1.5, label='0.1 LD ~ 38 400 km')
ax1.axvline(t_min_dias, color='#F44336', ls='-.', lw=2, alpha=0.8)
ax1.scatter([t_min_dias], [d_min_LD], s=150, zorder=5, color='#F44336',
            label=f'Minima: {d_min_km:.0f} km ({d_min_LD:.3f} LD)')
ax1.set_ylabel('Distancia [LD]', fontsize=12)
ax1.set_title('Aproximacion de Apophis a la Tierra -- 2029 (SSB, 5 cuerpos, IAS15)', fontsize=12)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, T_total_dias)

ax2 = axes[1]
ax2.plot(t_dias, v_rel_km_s, color='#4CAF50', lw=2, label='Velocidad relativa Apophis-Tierra')
ax2.axvline(t_min_dias, color='#F44336', ls='-.', lw=2, alpha=0.8)
ax2.scatter([t_min_dias], [v_flyby], s=150, zorder=5, color='#FF5722',
            label=f'Velocidad en flyby: {v_flyby:.2f} km/s')
ax2.set_xlabel('Dias desde 1 de abril de 2029', fontsize=12)
ax2.set_ylabel('Velocidad relativa [km/s]', fontsize=12)
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, T_total_dias)

plt.tight_layout()
plt.savefig('fig1_distancia_tiempo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: fig1_distancia_tiempo.png')



---

## 5. Trayectorias en el marco baricéntrico (SSB)

### 5.1 Vista del plano eclíptico

Visualizamos las órbitas de todos los cuerpos en el plano **x-y** del marco SSB durante los 30 días de simulación.

La "deriva" del Sol respecto al origen refleja que el baricentro del Sistema Solar no coincide exactamente con el centro del Sol (se desplaza ~1-2 R_sol dependiendo de la posición de Júpiter).


In [ ]:
# ── BLOQUE 6 – Figura 2: Orbitas en el plano ecl. (SSB) ────────────────────

colores_p = ['#FDD835', '#FF8F00', '#1565C0', '#B0BEC5', '#E53935']
tamanios  = [15, 10, 8, 4, 4]
zords     = [5, 4, 4, 3, 6]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, zoom, titulo in zip(axes,
                             [False, True],
                             ['Sistema completo (SSB)',
                              'Zoom: region del flyby']):
    ax.set_facecolor('#0A0A1A')
    fig.patch.set_facecolor('#0A0A1A')
    for j, (nom, col, sz, zo) in enumerate(zip(nombres, colores_p, tamanios, zords)):
        ax.plot(pos[:, j, 0], pos[:, j, 1], color=col,
                lw=2.5 if j==4 else 1.2, alpha=0.7, zorder=zo)
        ax.scatter(pos[-1, j, 0], pos[-1, j, 1], s=sz*8, color=col,
                   label=nom, edgecolors='white', linewidths=0.5, zorder=zo+1)
    ax.scatter(0, 0, marker='+', s=200, color='white', zorder=10, linewidths=2)
    ax.set_xlabel('x [UA]', fontsize=11); ax.set_ylabel('y [UA]', fontsize=11)
    ax.set_title(titulo, fontsize=12)
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.grid(True, alpha=0.15, color='white')
    if zoom:
        cx = np.mean(pos[:, I_EAR, 0])
        cy = np.mean(pos[:, I_EAR, 1])
        ax.set_xlim(cx - 0.005, cx + 0.005)
        ax.set_ylim(cy - 0.005, cy + 0.005)
    else:
        ax.legend(loc='upper left', fontsize=9,
                  facecolor='#111', edgecolor='#555', labelcolor='white')

plt.tight_layout()
plt.savefig('fig2_orbitas_SSB.png', dpi=150, bbox_inches='tight', facecolor='#0A0A1A')
plt.show()
print('Figura guardada: fig2_orbitas_SSB.png')



---

## 6. Verificación de leyes de conservación

Un indicador de la calidad de la integración es la conservación de la **energía total** y el **momento angular total**. Para IAS15 esperamos:

```
|delta_E / E_0| < 1e-9
```

Para un integrador simpléctico (WHFast) la energía *oscila* alrededor de un valor constante sin deriva secular.


In [ ]:
# ── BLOQUE 7 – Conservacion de energia y momento angular ───────────────────

def energia_total(pos_s, vel_s, masas, G=4*np.pi**2):
    N  = len(masas)
    Ek = 0.5 * np.sum(masas[:, None] * vel_s**2)
    Ep = 0.0
    for i in range(N):
        for j in range(i+1, N):
            rij = np.linalg.norm(pos_s[i] - pos_s[j])
            Ep -= G * masas[i] * masas[j] / rij
    return Ek + Ep

def Lz_total(pos_s, vel_s, masas):
    return sum(masas[i]*(pos_s[i,0]*vel_s[i,1]-pos_s[i,1]*vel_s[i,0])
               for i in range(len(masas)))

masas_sim = np.array([p.m for p in sim.particles])
stride    = 6
idx_sub   = np.arange(0, N_steps, stride)

E_arr  = np.array([energia_total(pos[i], vel[i], masas_sim) for i in idx_sub])
Lz_arr = np.array([Lz_total(pos[i], vel[i], masas_sim)     for i in idx_sub])

dE_rel  = (E_arr  - E_arr[0])  / abs(E_arr[0])
dLz_rel = (Lz_arr - Lz_arr[0]) / abs(Lz_arr[0])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7))
ax1.semilogy(t_dias[idx_sub], np.abs(dE_rel),  color='#E91E63', lw=1.5)
ax1.axhline(1e-9, ls='--', color='#888', lw=1, label='epsilon = 1e-9 (IAS15)')
ax1.set_ylabel('|delta_E / E_0|', fontsize=12)
ax1.set_title('Conservacion de energia -- IAS15 (5 cuerpos, 30 dias)', fontsize=12)
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.semilogy(t_dias[idx_sub], np.abs(dLz_rel)+1e-20, color='#00BCD4', lw=1.5)
ax2.set_ylabel('|delta_Lz / Lz_0|', fontsize=12)
ax2.set_xlabel('Dias desde 1 de abril de 2029', fontsize=11)
ax2.set_title('Conservacion de momento angular -- IAS15', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig3_conservacion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Error max. en E  : {np.abs(dE_rel).max():.2e}')
print(f'Error max. en Lz : {np.abs(dLz_rel).max():.2e}')
print('-> Integracion correcta: errores por debajo de la tolerancia.')



---

## 7. Efecto de Júpiter y la Luna sobre el flyby

Un punto científicamente relevante es cuánto *afectan* los cuerpos secundarios a la geometría del encuentro. Corremos simulaciones alternativas:

1. **Solo Sol + Tierra + Apophis** (3 cuerpos)
2. **Sol + Júpiter + Tierra + Apophis** (sin Luna)
3. **Sol + Tierra + Luna + Apophis** (sin Júpiter)
4. **Caso completo: 5 cuerpos** (referencia)


In [ ]:
# ── BLOQUE 8 – Efecto de Jupiter y la Luna (4 escenarios) ──────────────────

def simular_escenario(inc_jupiter=True, inc_luna=True,
                      epoch=EPOCH, n_steps=N_steps, dt_yr=dt_output_yr):
    s = rebound.Simulation()
    s.units = ('AU', 'yr', 'Msun')
    s.integrator = 'ias15'
    s.ri_ias15.epsilon = 1e-9
    cuerpos = [('Sun','sun',None), ('Earth','399',None),
               ('Apophis','2099942',1.9e-17)]
    if inc_jupiter: cuerpos.insert(1, ('Jupiter','5',None))
    if inc_luna:    cuerpos.insert(-1, ('Moon','301',None))
    for nm, hid, mass in cuerpos:
        p = rebound.Particle(simulation=s, primary=None, date=epoch, id=hid)
        if mass is not None: p.m = mass
        p.hash = nm
        s.add(p)
    s.move_to_com()
    idx_e = next(i for i,p in enumerate(s.particles)
                 if p.hash==rebound.hash('Earth'))
    idx_a = next(i for i,p in enumerate(s.particles)
                 if p.hash==rebound.hash('Apophis'))
    d_min = np.inf
    for _ in range(n_steps):
        s.integrate(s.t + dt_yr)
        pe = s.particles[idx_e]; pa = s.particles[idx_a]
        d = ((pa.x-pe.x)**2+(pa.y-pe.y)**2+(pa.z-pe.z)**2)**0.5
        if d < d_min: d_min = d
    return d_min * 1.496e8

print('Ejecutando 4 escenarios...')
d_esc = {}
d_esc['Solo Sol+Tierra (3 cuerpos)'] = simular_escenario(False, False)
d_esc['Sol+Jupiter+Tierra']          = simular_escenario(True,  False)
d_esc['Sol+Tierra+Luna']             = simular_escenario(False, True)
d_esc['5 cuerpos (completo)']        = d_min_km

print('\n' + '='*55)
print('  EFECTO DE JUPITER Y LUNA SOBRE LA DISTANCIA MINIMA')
print('='*55)
for nombre, d in d_esc.items():
    delta = d - d_min_km
    print(f'  {nombre:<28}: {d:>10.1f} km  (delta={delta:+.1f})')
print('='*55)



In [ ]:
# ── BLOQUE 9 – Figura 4: Comparacion de escenarios ────────────────────────

labels   = list(d_esc.keys())
valores  = np.array(list(d_esc.values()))
colores_b = ['#607D8B', '#FF9800', '#03A9F4', '#4CAF50']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(labels, valores/1e3, color=colores_b, edgecolor='white', height=0.55)
ax.axvline(d_min_km/1e3, color='#F44336', ls='--', lw=1.5,
           label=f'Referencia 5 cuerpos: {d_min_km/1e3:.1f} x 10^3 km')
for bar, val in zip(bars, valores):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'{val/1e3:.1f} x 10^3 km', va='center', fontsize=10)
ax.set_xlabel('Distancia minima Tierra-Apophis [10^3 km]', fontsize=11)
ax.set_title('Influencia de Jupiter y la Luna en el flyby 2029', fontsize=12)
ax.legend(fontsize=10); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('fig4_escenarios.png', dpi=150, bbox_inches='tight')
plt.show()



---

## 8. Parámetros del encuentro y geometría hiperbólica

### 8.1 Hipérbola geocéntrica

Durante el encuentro, si v_inf > v_esc de la Tierra, la trayectoria es **hiperbólica** en el marco geocéntrico:

```
Semieje hiperbólico:  a_h = -G*M_Tierra / v_inf^2   (negativo)
Excentricidad:        e_h = 1 + v_inf^2 * d_min / (G*M_Tierra)
Angulo de deflexion:  theta = 2 * arcsin(1/e_h)
```

El parámetro de impacto `b` es la distancia de máxima aproximación *proyectada* que tendría la trayectoria si no hubiera gravedad.


In [ ]:
# ── BLOQUE 10 – Parametros hiperbolicos del flyby ──────────────────────────

GM_earth = 398600.4418   # km^3/s^2
R_earth  = 6371.0        # km

v_inf_km_s = v_rel_km_s[0]                    # velocidad al inicio (lejos del flyby)
v_esc      = (2*GM_earth/R_earth)**0.5         # km/s en la superficie
a_h_km     = -GM_earth / v_inf_km_s**2        # km (negativo: hiperbola)
e_h        = 1 + v_inf_km_s**2 * d_min_km / GM_earth
b_km       = abs(a_h_km) * (e_h**2 - 1)**0.5
theta_deg  = np.degrees(2 * np.arcsin(1/e_h))
v_peri     = (v_inf_km_s**2 + 2*GM_earth/d_min_km)**0.5

print('='*55)
print('  PARAMETROS DE LA HIPERBOL GEOCENTRICA -- 2029')
print('='*55)
print(f'  v_inf (lejos del flyby)  : {v_inf_km_s:.3f} km/s')
print(f'  v_esc Tierra (superficie): {v_esc:.3f} km/s')
print(f'  v_periapsis              : {v_peri:.3f} km/s')
print(f'  v_inf / v_esc            : {v_inf_km_s/v_esc:.3f}')
print(f'  |a_h|                    : {abs(a_h_km):.0f} km')
print(f'  Excentricidad e_h        : {e_h:.4f}')
print(f'  Param. de impacto b      : {b_km:.0f} km')
print(f'  Distancia minima d_min   : {d_min_km:.0f} km')
print(f'  d_min / R_Tierra         : {d_min_km/R_earth:.2f}')
print(f'  Angulo de deflexion      : {theta_deg:.4f} grados')
print('='*55)



---

## 9. Marco geocéntrico: trayectoria de Apophis vista desde la Tierra

En el marco geocéntrico, Apophis sigue una **hipérbola** alrededor de la Tierra (cuando domina el campo gravitacional terrestre). 

### Por qué cambiar de referencia

En el marco **SSB** la escala del movimiento orbital es ~1 UA, mientras que el flyby ocurre en ~0.0003 UA. El detalle del encuentro se aprecia mucho mejor en el marco geocéntrico donde:

- La **Tierra está fija** en el origen.
- Apophis describe una hipérbola cuyo periapsis es la distancia de máxima aproximación.
- La Luna contextualiza la escala (su distancia media es ~384 400 km = 1 LD).


In [ ]:
# ── BLOQUE 11 – Marco geocentrico: vectores relativos ──────────────────────

r_APO_geo = pos[:, I_APO, :] - pos[:, I_EAR, :]   # Apophis geocentrico
r_MOO_geo = pos[:, I_MOO, :] - pos[:, I_EAR, :]   # Luna geocentrica
r_SUN_geo = pos[:, I_SUN, :] - pos[:, I_EAR, :]   # Sol geocentrico

d_moo_km = np.linalg.norm(r_MOO_geo, axis=1) * 1.496e8

print('Estadisticas geocentricas:')
print(f'  Distancia Luna-Tierra (media)  : {d_moo_km.mean()/1e3:.0f} x 10^3 km')
print(f'                                 : {d_moo_km.mean()/384400:.4f} LD')
print(f'  Distancia Apophis (minima)     : {d_min_km:.0f} km')
print(f'  Escala flyby / escala lunar    : {d_min_km/d_moo_km.mean():.4f}')



In [ ]:
# ── BLOQUE 12 – Figura 5: Trayectoria geocentrica (XY y XZ) ────────────────

fig = plt.figure(figsize=(16, 8))
fig.patch.set_facecolor('#0D1117')
gs  = GridSpec(1, 2, figure=fig)

for ax_idx, (i1, i2, xlabel, ylabel, titulo) in enumerate([
    (0, 1, 'x geocentrico [10^3 km]', 'y geocentrico [10^3 km]', 'Plano eclipt. geocentrico (X-Y)'),
    (0, 2, 'x geocentrico [10^3 km]', 'z geocentrico [10^3 km]', 'Plano meridional geocentrico (X-Z)'),
]):
    ax = fig.add_subplot(gs[0, ax_idx])
    ax.set_facecolor('#060A10')

    xa = r_APO_geo[:, i1] * AU2kkm
    ya = r_APO_geo[:, i2] * AU2kkm
    t_norm = (t_dias - t_dias[0]) / (t_dias[-1] - t_dias[0])
    sc = ax.scatter(xa, ya, c=t_norm, cmap='plasma', s=6, zorder=4, alpha=0.9)

    xl = r_MOO_geo[:, i1] * AU2kkm
    yl = r_MOO_geo[:, i2] * AU2kkm
    ax.plot(xl, yl, color='#B0BEC5', lw=1, alpha=0.5, label='Orbita lunar')

    ax.scatter(0, 0, s=300, color='#1565C0', zorder=8, label='Tierra')
    d_luna_medio = np.mean(np.linalg.norm(r_MOO_geo, axis=1)) * AU2kkm
    circ_l = plt.Circle((0,0), d_luna_medio, color='#888', ls='--', fill=False, alpha=0.4)
    ax.add_patch(circ_l)

    i_min_geo = np.argmin(np.linalg.norm(r_APO_geo, axis=1))
    ax.scatter(r_APO_geo[i_min_geo, i1]*AU2kkm, r_APO_geo[i_min_geo, i2]*AU2kkm,
               s=200, color='#F44336', zorder=9, marker='*',
               label=f'Periapsis: {d_min_km:.0f} km')

    cb = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.04)
    cb.set_label('Tiempo normalizado', fontsize=9)
    ax.set_xlabel(xlabel, fontsize=11); ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(titulo, fontsize=12)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
    ax.title.set_color('white')

plt.suptitle('Marco geocentrico -- Flyby Apophis 2029', color='white', fontsize=14)
plt.tight_layout()
plt.savefig('fig5_geocentrico.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()



In [ ]:
# ── BLOQUE 13 – Figura 6: Zoom sobre el flyby geocentrico ───────────────────
# Plot detallado de Tierra-Apophis durante el acercamiento

mask_flyby = np.abs(t_dias - t_min_dias) <= 3
t_fly   = t_dias[mask_flyby]
xa_fly  = r_APO_geo[mask_flyby, 0] * AU2kkm
ya_fly  = r_APO_geo[mask_flyby, 1] * AU2kkm

fig, ax = plt.subplots(figsize=(10, 9))
ax.set_facecolor('#060A10'); fig.patch.set_facecolor('#060A10')

sc = ax.scatter(xa_fly, ya_fly, c=t_fly, cmap='RdYlGn_r',
                s=30, zorder=5, alpha=0.95, edgecolors='none')
cb = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.04)
cb.set_label('Dias desde 1 abr 2029', fontsize=10, color='white')
cb.ax.yaxis.set_tick_params(color='white')
plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

ax.scatter(0, 0, s=500, color='#2196F3', zorder=10, label='Tierra')
circ_t = plt.Circle((0,0), R_earth/1e3, color='#1565C0', alpha=0.5, zorder=9)
ax.add_patch(circ_t)

i_min_fly = np.argmin(np.linalg.norm(r_APO_geo[mask_flyby,:2], axis=1))
ax.scatter(xa_fly[i_min_fly], ya_fly[i_min_fly], s=350, color='#FF5722',
           zorder=11, marker='*', label=f'Periapsis: {d_min_km:.0f} km')
ax.annotate(f'Periapsis\n{d_min_km:.0f} km\nt={t_min_dias:.2f} d',
            xy=(xa_fly[i_min_fly], ya_fly[i_min_fly]),
            xytext=(xa_fly[i_min_fly]+25, ya_fly[i_min_fly]+25),
            color='white', fontsize=10,
            arrowprops=dict(arrowstyle='->', color='#FF5722'))

for d_ref, etiqueta in [(50000,'50 000 km'),(100000,'100 000 km'),(384400,'1 LD')]:
    c = plt.Circle((0,0), d_ref/1e3, ls=':', fill=False, color='#555', alpha=0.6)
    ax.add_patch(c)
    ax.text(d_ref/1e3*0.707, d_ref/1e3*0.707, etiqueta, color='#888', fontsize=8)

ax.set_xlim(-250, 250); ax.set_ylim(-250, 250)
ax.set_xlabel('x geocentrico [10^3 km]', fontsize=12, color='white')
ax.set_ylabel('y geocentrico [10^3 km]', fontsize=12, color='white')
ax.set_title('Trayectoria geocentrica de Apophis -- +-3 dias del flyby (2029)',
             fontsize=13, color='white')
ax.tick_params(colors='white')
ax.legend(fontsize=11, facecolor='#111', edgecolor='#555', labelcolor='white')
ax.grid(True, alpha=0.15, color='white')
plt.tight_layout()
plt.savefig('fig6_flyby_zoom.png', dpi=150, bbox_inches='tight', facecolor='#060A10')
plt.show()
print('Figura guardada: fig6_flyby_zoom.png')



In [ ]:
# ── BLOQUE 14 – Figura 7: Componentes de velocidad geocentrica ──────────────

vr_geo = (vel[:, I_APO, :] - vel[:, I_EAR, :]) * AU_yr_to_km_s
vx_geo = vr_geo[:, 0]; vy_geo = vr_geo[:, 1]; vz_geo = vr_geo[:, 2]
v_total_geo = np.linalg.norm(vr_geo, axis=1)

r_unit  = r_APO_geo / (np.linalg.norm(r_APO_geo, axis=1, keepdims=True) + 1e-30)
v_radial = np.sum(vr_geo * r_unit, axis=1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax1.plot(t_dias, vx_geo, label='vx', color='#E91E63', lw=1.5)
ax1.plot(t_dias, vy_geo, label='vy', color='#00BCD4', lw=1.5)
ax1.plot(t_dias, vz_geo, label='vz', color='#FF9800', lw=1.5)
ax1.plot(t_dias, v_total_geo, label='|v| total', color='white', lw=2, ls='--')
ax1.axvline(t_min_dias, color='#F44336', ls='-.',  alpha=0.7)
ax1.set_ylabel('Velocidad geocentrica [km/s]', fontsize=11)
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)
ax1.set_title('Velocidad geocentrica de Apophis -- Marco geocentrico', fontsize=12)

ax2.plot(t_dias, v_radial, color='#7E57C2', lw=2, label='v radial (+ = alejandose)')
ax2.axhline(0, color='gray', ls='--', lw=1)
ax2.axvline(t_min_dias, color='#F44336', ls='-.', alpha=0.7,
            label=f'Flyby en t={t_min_dias:.2f} dias')
ax2.set_xlabel('Dias desde 1 de abril de 2029', fontsize=11)
ax2.set_ylabel('Velocidad radial [km/s]', fontsize=11)
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig7_velocidades.png', dpi=150, bbox_inches='tight')
plt.show()



---

## 10. Animaciones con Celluloid

### ¿Qué es Celluloid?

**Celluloid** [6] es una librería Python que simplifica la creación de animaciones con Matplotlib. En lugar de gestionar manualmente `FuncAnimation`, Celluloid usa la metáfora de una **cámara fotográfica**:

```python
from celluloid import Camera

fig, ax = plt.subplots()
camera = Camera(fig)       # 1. Crear camara

for frame in data:
    ax.plot(...)           # 2. Dibujar cada fotograma
    camera.snap()          # 3. Fotografiar

anim = camera.animate()    # 4. Crear animacion
anim.save("anim.gif")      # 5. Guardar GIF/MP4
```

**Ventajas:**
- No requiere `update()` ni `init()` como `FuncAnimation`.
- Funciona con cualquier tipo de gráfico.
- Compatible con `HTML(anim.to_jshtml())` para embeber en Jupyter.

**Referencia:**  
[6] Morris, J. (2019). *Celluloid: Simple animation in Matplotlib*. GitHub: https://github.com/jwkvam/celluloid

### Animaciones que produciremos

1. **Animación SSB**: Sistema completo (5 cuerpos) en el plano eclíptico.
2. **Animación geocéntrica**: Trayectoria de Apophis alrededor de la Tierra durante el flyby.


In [ ]:
# ── BLOQUE 15 – Animacion 1: Sistema completo en el SSB (Celluloid) ─────────
# Ref: Morris (2019) -- https://github.com/jwkvam/celluloid

n_frames_a1 = 60
idx_anim_a1 = np.linspace(0, N_steps-1, n_frames_a1, dtype=int)

fig_a1, ax_a1 = plt.subplots(figsize=(9, 9))
fig_a1.patch.set_facecolor('#0A0A1A')
ax_a1.set_facecolor('#0A0A1A')
camera_a1 = Camera(fig_a1)

colores_a  = ['#FDD835', '#FF8F00', '#1565C0', '#B0BEC5', '#E53935']
tamanios_a = [200, 80, 60, 20, 20]

for fi in idx_anim_a1:
    ax_a1.cla()
    ax_a1.set_facecolor('#0A0A1A')
    trail = max(0, fi - 48)
    for j, (nom, col) in enumerate(zip(nombres, colores_a)):
        ax_a1.plot(pos[trail:fi+1, j, 0], pos[trail:fi+1, j, 1],
                   color=col, lw=1.2, alpha=0.5)
        ax_a1.scatter(pos[fi, j, 0], pos[fi, j, 1],
                      s=tamanios_a[j], color=col, label=nom,
                      edgecolors='white', linewidths=0.3, zorder=5)
    ax_a1.plot([pos[fi,I_EAR,0], pos[fi,I_APO,0]],
               [pos[fi,I_EAR,1], pos[fi,I_APO,1]],
               color='#FF5722', ls='--', lw=0.8, alpha=0.6)
    ax_a1.scatter(0, 0, marker='+', s=100, color='white', lw=2, zorder=10)
    ax_a1.text(0.02, 0.97,
               f't = {t_dias[fi]:.1f} dias\nd(Tierra-Apophis) = {d_EA_LD[fi]:.4f} LD',
               transform=ax_a1.transAxes, color='white', fontsize=10,
               va='top', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='#111', alpha=0.7))
    ax_a1.legend(loc='upper right', fontsize=8,
                 facecolor='#111', edgecolor='#555', labelcolor='white')
    ax_a1.set_xlim(-1.3, 1.3); ax_a1.set_ylim(-1.3, 1.3)
    ax_a1.set_xlabel('x [UA]', color='white')
    ax_a1.set_ylabel('y [UA]', color='white')
    ax_a1.set_title('Sistema Solar (SSB) -- Flyby Apophis 2029', color='white', fontsize=12)
    ax_a1.tick_params(colors='white')
    ax_a1.grid(True, alpha=0.1, color='white')
    camera_a1.snap()

print('Compilando animacion SSB...')
anim_ssb = camera_a1.animate(interval=80, blit=False)
anim_ssb.save('anim_ssb.gif', writer='pillow', fps=12, dpi=100)
print('Guardada: anim_ssb.gif')
display(HTML(anim_ssb.to_jshtml(fps=12)))



In [ ]:
# ── BLOQUE 16 – Animacion 2: Flyby en el marco geocentrico (Celluloid) ──────

mask_geo_a = np.abs(t_dias - t_min_dias) <= 4
idx_geo    = np.where(mask_geo_a)[0]
n_frames_a2 = 80
idx_geo_sub = idx_geo[np.linspace(0, len(idx_geo)-1, n_frames_a2, dtype=int)]

fig_a2, ax_a2 = plt.subplots(figsize=(9, 9))
fig_a2.patch.set_facecolor('#060A10')
ax_a2.set_facecolor('#060A10')
camera_a2 = Camera(fig_a2)

for count, fi in enumerate(idx_geo_sub):
    ax_a2.cla()
    ax_a2.set_facecolor('#060A10')

    trail_start = idx_geo_sub[max(0, count-40)]
    trail_mask  = np.arange(trail_start, fi+1)
    ax_a2.plot(r_APO_geo[trail_mask, 0]*AU2kkm,
               r_APO_geo[trail_mask, 1]*AU2kkm,
               color='#E53935', lw=1.5, alpha=0.7)
    ax_a2.scatter(r_APO_geo[fi, 0]*AU2kkm, r_APO_geo[fi, 1]*AU2kkm,
                  s=120, color='#E53935', zorder=7, edgecolors='white',
                  linewidths=0.5, label='Apophis')

    ax_a2.scatter(r_MOO_geo[fi, 0]*AU2kkm, r_MOO_geo[fi, 1]*AU2kkm,
                  s=40, color='#B0BEC5', zorder=6, edgecolors='white',
                  linewidths=0.3, label='Luna')
    ax_a2.plot(r_MOO_geo[:fi+1, 0]*AU2kkm, r_MOO_geo[:fi+1, 1]*AU2kkm,
               color='#607D8B', lw=0.8, alpha=0.3)

    ax_a2.scatter(0, 0, s=350, color='#1565C0', zorder=8, label='Tierra')

    for d_ref, lab in [(50000,'50 Mm'), (100000,'100 Mm'), (384400,'1 LD')]:
        c = plt.Circle((0,0), d_ref/1e3, ls=':', fill=False, color='#334', alpha=0.7)
        ax_a2.add_patch(c)
        ax_a2.text(d_ref/1e3*0.707, d_ref/1e3*0.707, lab, color='#446', fontsize=8)

    d_act_km = d_EA_km[fi]
    ax_a2.text(0.02, 0.97,
               f't = {t_dias[fi]:.2f} dias\nd = {d_act_km:.0f} km\n  = {d_act_km/384400:.4f} LD',
               transform=ax_a2.transAxes, color='white', fontsize=10,
               va='top', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='#111', alpha=0.8))
    ax_a2.set_xlim(-500, 500); ax_a2.set_ylim(-500, 500)
    ax_a2.set_xlabel('x geocentrico [10^3 km]', color='white', fontsize=11)
    ax_a2.set_ylabel('y geocentrico [10^3 km]', color='white', fontsize=11)
    ax_a2.set_title('Marco geocentrico -- Flyby Apophis 2029', color='white', fontsize=12)
    ax_a2.tick_params(colors='white')
    ax_a2.grid(True, alpha=0.12, color='white')
    ax_a2.legend(loc='upper right', fontsize=9,
                 facecolor='#111', edgecolor='#555', labelcolor='white')
    camera_a2.snap()

print('Compilando animacion geocentrica...')
anim_geo = camera_a2.animate(interval=80, blit=False)
anim_geo.save('anim_geocentrico.gif', writer='pillow', fps=12, dpi=100)
print('Guardada: anim_geocentrico.gif')
display(HTML(anim_geo.to_jshtml(fps=12)))



---

## 11. Análisis de Monte Carlo: sensibilidad a incertidumbres iniciales

Las efemérides de Apophis tienen **incertidumbres observacionales**. Aunque estas son muy pequeñas en 2029 (gracias a décadas de observaciones radar y ópticas), es didáctico evaluar cuánto cambia la distancia de máxima aproximación cuando perturbamos las condiciones iniciales.

### Incertidumbre típica

Para un objeto bien caracterizado como Apophis (radar Arecibo/Goldstone, OSIRIS-APEX), las incertidumbres en posición son ~10-100 km y en velocidad ~1-10 mm/s [7].

Corremos **N = 100 clones** con condiciones iniciales perturbadas aleatoriamente:

```
r_clone = r_Apophis + Normal(0, sigma_r)
v_clone = v_Apophis + Normal(0, sigma_v)
```

[7] Farnocchia D. et al. (2023) *AJ* 165, 238.


In [ ]:
# ── BLOQUE 17 – Monte Carlo (N=100 clones de Apophis) ──────────────────────

sigma_r_km = 50.0
sigma_v_mms = 5.0
sigma_r = sigma_r_km / 1.496e8
sigma_v = sigma_v_mms * 1e-3 / (1.496e8 / (365.25*86400))

N_MC = 100
rng  = np.random.default_rng(42)
d_min_mc = np.zeros(N_MC)

p_apo_ref = sim.particles['Apophis']
r_nom = np.array([p_apo_ref.x, p_apo_ref.y, p_apo_ref.z])
v_nom = np.array([p_apo_ref.vx, p_apo_ref.vy, p_apo_ref.vz])

print(f'Corriendo {N_MC} clones (sigma_r={sigma_r_km} km, sigma_v={sigma_v_mms} mm/s)...')

for mc in range(N_MC):
    dr = rng.normal(0, sigma_r, 3)
    dv = rng.normal(0, sigma_v, 3)
    s_mc = sim.copy()
    apo_mc = s_mc.particles['Apophis']
    apo_mc.x  += dr[0]; apo_mc.y  += dr[1]; apo_mc.z  += dr[2]
    apo_mc.vx += dv[0]; apo_mc.vy += dv[1]; apo_mc.vz += dv[2]
    s_mc.move_to_com()
    idx_e = next(i for i,p in enumerate(s_mc.particles) if p.hash==rebound.hash('Earth'))
    idx_a = next(i for i,p in enumerate(s_mc.particles) if p.hash==rebound.hash('Apophis'))
    d_min_this = np.inf
    for _ in range(N_steps):
        s_mc.integrate(s_mc.t + dt_output_yr)
        pe = s_mc.particles[idx_e]; pa = s_mc.particles[idx_a]
        d = ((pa.x-pe.x)**2+(pa.y-pe.y)**2+(pa.z-pe.z)**2)**0.5
        if d < d_min_this: d_min_this = d
    d_min_mc[mc] = d_min_this * 1.496e8

print(f'Completado.')
print(f'  Media   : {d_min_mc.mean():.0f} km')
print(f'  Mediana : {np.median(d_min_mc):.0f} km')
print(f'  Std     : {d_min_mc.std():.0f} km')
print(f'  Min/Max : {d_min_mc.min():.0f} / {d_min_mc.max():.0f} km')



In [ ]:
# ── BLOQUE 18 – Figura 8: Histograma Monte Carlo + CDF ─────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.hist(d_min_mc/1e3, bins=20, color='#7E57C2', edgecolor='white', alpha=0.85, density=True)
ax1.axvline(d_min_km/1e3, color='#F44336', lw=2, ls='--',
            label=f'Nominal: {d_min_km/1e3:.1f} x 10^3 km')
ax1.axvline(d_min_mc.mean()/1e3, color='#FF9800', lw=2, ls='-.',
            label=f'Media MC: {d_min_mc.mean()/1e3:.1f} x 10^3 km')
ax1.set_xlabel('Distancia minima [10^3 km]', fontsize=11)
ax1.set_ylabel('Densidad de probabilidad', fontsize=11)
ax1.set_title(f'Distribucion Monte Carlo (N={N_MC})', fontsize=11)
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)

d_sorted = np.sort(d_min_mc)/1e3
cdf = np.arange(1, len(d_sorted)+1)/len(d_sorted)
ax2.plot(d_sorted, cdf, color='#00BCD4', lw=2)
ax2.axvline(d_min_km/1e3, color='#F44336', lw=2, ls='--',
            label=f'Nominal: {d_min_km/1e3:.1f}')
for pct, label in [(0.05,'5%'),(0.50,'50%'),(0.95,'95%')]:
    ax2.axhline(pct, color='#888', ls=':', lw=1, label=f'Percentil {label}')
ax2.set_xlabel('Distancia minima [10^3 km]', fontsize=11)
ax2.set_ylabel('Probabilidad acumulada', fontsize=11)
ax2.set_title('CDF -- distancia minima Tierra-Apophis', fontsize=11)
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.suptitle('Monte Carlo -- Sensibilidad del flyby a las condiciones iniciales',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig8_montecarlo.png', dpi=150, bbox_inches='tight')
plt.show()



---

## 12. Evolución de los elementos orbitales de Apophis

Durante el flyby, el campo gravitacional terrestre **modifica los elementos orbitales heliocéntricos** de Apophis. Este es el corazón del fenómeno del *keyhole* (agujero de llave): dependiendo de por dónde exactamente pase Apophis en 2029, los elementos resultantes pueden o no colocar a Apophis en resonancia con la Tierra para 2036 o 2068.

Calculamos los elementos orbitales de Apophis respecto al Sol usando la energía específica y el momento angular de la órbita heliocéntrica.


In [ ]:
# ── BLOQUE 19 – Elementos orbitales helio. de Apophis durante el flyby ──────

a_arr = []; e_arr = []; inc_arr = []; t_elem = []
GM_sun = 4*np.pi**2   # UA^3/yr^2 (G=1, M_sol=1)

for i in range(0, N_steps, 6):
    r_vec = pos[i, I_APO, :] - pos[i, I_SUN, :]
    v_vec = vel[i, I_APO, :] - vel[i, I_SUN, :]
    r  = np.linalg.norm(r_vec)
    v2 = np.sum(v_vec**2)
    eps = 0.5*v2 - GM_sun/r
    a_  = -GM_sun / (2*eps) if eps != 0 else np.inf
    h   = np.cross(r_vec, v_vec)
    h_m = np.linalg.norm(h)
    disc = 1 + 2*eps*h_m**2/GM_sun**2
    e_  = disc**0.5 if disc >= 0 else 0.0
    inc_ = np.degrees(np.arccos(np.clip(h[2]/h_m, -1, 1))) if h_m > 0 else 0.0
    a_arr.append(a_); e_arr.append(e_); inc_arr.append(inc_)
    t_elem.append(t_dias[i])

a_arr=np.array(a_arr); e_arr=np.array(e_arr)
inc_arr=np.array(inc_arr); t_elem=np.array(t_elem)

print('Elementos orbitales de Apophis:')
print(f'  Antes del flyby  -- a={a_arr[0]:.5f} UA, e={e_arr[0]:.5f}, i={inc_arr[0]:.3f} deg')
print(f'  Despues del flyby-- a={a_arr[-1]:.5f} UA, e={e_arr[-1]:.5f}, i={inc_arr[-1]:.3f} deg')
da = a_arr[-1] - a_arr[0]
de = e_arr[-1] - e_arr[0]
print(f'  delta_a = {da:+.5f} UA = {da*1.496e8:.0f} km')
print(f'  delta_e = {de:+.5f}')



In [ ]:
# ── BLOQUE 20 – Figura 9: Evolucion de elementos orbitales ─────────────────

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

for ax, data, ylabel, col in [
    (axes[0], a_arr,   'Semieje mayor a [UA]', '#FDD835'),
    (axes[1], e_arr,   'Excentricidad e',      '#E91E63'),
    (axes[2], inc_arr, 'Inclinacion i [deg]',  '#00BCD4'),
]:
    ax.plot(t_elem, data, color=col, lw=2)
    ax.axvline(t_min_dias, color='#F44336', ls='-.', lw=1.5, label=f'Flyby t={t_min_dias:.2f} d')
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[2].set_xlabel('Dias desde 1 de abril de 2029', fontsize=11)
plt.suptitle('Evolucion elementos orbitales heliocentricos de Apophis durante el flyby 2029',
             fontsize=12)
plt.tight_layout()
plt.savefig('fig9_elementos_orbitales.png', dpi=150, bbox_inches='tight')
plt.show()



---

## 13. Comparación de integradores: IAS15 vs WHFast

REBOUND ofrece múltiples integradores. Para encuentros cercanos, la elección es crucial:

| Integrador | Tipo | Paso adaptativo | Recomendado para encuentros |
|---|---|---|---|
| **IAS15** | Implícito, orden 15 | Si | **Si** |
| **WHFast** | Simpléctico, orden 2 | No (paso fijo) | No (puede errar) |
| **MERCURIUS** | Híbrido WHFast/IAS15 | Parcial | Si (N-body general) |

Compararemos la distancia mínima calculada por IAS15 y WHFast con diferentes pasos fijos.


In [ ]:
# ── BLOQUE 21 – Comparacion IAS15 vs WHFast ─────────────────────────────────

def dmin_integrador(integrador='ias15', dt_fijo_min=15.0):
    s = rebound.Simulation()
    s.units = ('AU', 'yr', 'Msun')
    s.integrator = integrador
    for nm, hid, mass in [('Sun','sun',None),('Jupiter','5',None),
                           ('Earth','399',None),('Moon','301',None),
                           ('Apophis','2099942',1.9e-17)]:
        p = rebound.Particle(simulation=s, primary=None, date=EPOCH, id=hid)
        if mass: p.m = mass
        p.hash = nm
        s.add(p)
    s.move_to_com()
    if integrador == 'whfast':
        s.dt = dt_fijo_min / (365.25*24*60)
    idx_e = next(i for i,p in enumerate(s.particles) if p.hash==rebound.hash('Earth'))
    idx_a = next(i for i,p in enumerate(s.particles) if p.hash==rebound.hash('Apophis'))
    d_min = np.inf
    for _ in range(N_steps):
        s.integrate(s.t + dt_output_yr)
        pe=s.particles[idx_e]; pa=s.particles[idx_a]
        d = ((pa.x-pe.x)**2+(pa.y-pe.y)**2+(pa.z-pe.z)**2)**0.5
        if d < d_min: d_min = d
    return d_min * 1.496e8

print('IAS15 (referencia, adaptativo)...')
d_ias15_ref = d_min_km
print('WHFast (paso=15 min)...')
d_whf_15 = dmin_integrador('whfast', 15.0)
print('WHFast (paso=5 min)...')
d_whf_5  = dmin_integrador('whfast', 5.0)

print('\n' + '='*55)
print('  COMPARACION DE INTEGRADORES')
print('='*55)
print(f'  IAS15 (eps=1e-9)     : {d_ias15_ref:.1f} km')
print(f'  WHFast (dt=15 min)   : {d_whf_15:.1f} km  (delta={d_whf_15-d_ias15_ref:+.1f} km)')
print(f'  WHFast (dt=5 min)    : {d_whf_5:.1f} km  (delta={d_whf_5-d_ias15_ref:+.1f} km)')
print('='*55)
print('-> Para encuentros cercanos: usar siempre IAS15 o MERCURIUS.')



In [ ]:
# ── BLOQUE 22 – Figura 10: Panel resumen del flyby ──────────────────────────

fig = plt.figure(figsize=(18, 13))
fig.patch.set_facecolor('#0A0A1A')
gs_main = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

# (0,0-1) Distancia vs tiempo
ax00 = fig.add_subplot(gs_main[0, :2])
ax00.set_facecolor('#111')
ax00.plot(t_dias, d_EA_LD, color='#2196F3', lw=2)
ax00.axhline(1, color='#FF9800', ls='--', lw=1.2, label='1 LD')
ax00.axvline(t_min_dias, color='#F44336', ls='-.')
ax00.scatter([t_min_dias],[d_min_LD], s=150, color='#F44336', zorder=5,
             label=f'Min: {d_min_LD:.3f} LD')
ax00.set_title('Distancia Tierra-Apophis (SSB)', color='white', fontsize=11)
ax00.set_ylabel('Distancia [LD]', color='white')
ax00.tick_params(colors='white'); ax00.legend(fontsize=9,
    facecolor='#222', edgecolor='#555', labelcolor='white')
ax00.grid(True, alpha=0.2, color='white')

# (0,2) Tabla informativa
ax02 = fig.add_subplot(gs_main[0, 2])
ax02.set_facecolor('#111'); ax02.axis('off')
info = (f'FLYBY 2029 -- DATOS CLAVE\n\n'
        f'd_min : {d_min_km:.0f} km\n'
        f'       {d_min_LD:.4f} LD\n\n'
        f'v_flyby : {v_flyby:.2f} km/s\n\n'
        f'e_h : {e_h:.3f}\n\n'
        f'theta : {theta_deg:.4f} deg\n\n'
        f'delta_a : {da*1.496e8:.0f} km')
ax02.text(0.05, 0.95, info, transform=ax02.transAxes, color='white',
          fontsize=10, va='top', fontfamily='monospace',
          bbox=dict(boxstyle='round', facecolor='#1A237E', alpha=0.8))

# (1,0-1) Trayectoria geocentrica
ax10 = fig.add_subplot(gs_main[1, :2])
ax10.set_facecolor('#060A10')
sc10 = ax10.scatter(r_APO_geo[mask_flyby, 0]*AU2kkm,
                    r_APO_geo[mask_flyby, 1]*AU2kkm,
                    c=t_dias[mask_flyby], cmap='plasma', s=8, alpha=0.9)
ax10.scatter(0,0, s=300, color='#1565C0', zorder=8, label='Tierra')
ax10.scatter(r_APO_geo[i_min,0]*AU2kkm, r_APO_geo[i_min,1]*AU2kkm,
             s=200, color='#FF5722', zorder=9, marker='*',
             label=f'Periapsis: {d_min_km:.0f} km')
plt.colorbar(sc10, ax=ax10, fraction=0.025).set_label('Dias', color='white', fontsize=8)
ax10.set_title('Trayectoria geocentrica +/-3d flyby', color='white', fontsize=11)
ax10.set_xlabel('x [10^3 km]', color='white'); ax10.set_ylabel('y [10^3 km]', color='white')
ax10.tick_params(colors='white'); ax10.grid(True, alpha=0.15, color='white')
ax10.legend(fontsize=9, facecolor='#222', edgecolor='#555', labelcolor='white')
ax10.set_xlim(-200, 200); ax10.set_ylim(-200, 200)

# (1,2) Monte Carlo hist
ax12 = fig.add_subplot(gs_main[1, 2])
ax12.set_facecolor('#111')
ax12.hist(d_min_mc/1e3, bins=15, color='#7E57C2', edgecolor='white', alpha=0.85)
ax12.axvline(d_min_km/1e3, color='#F44336', lw=2, ls='--')
ax12.set_title('Monte Carlo d_min', color='white', fontsize=10)
ax12.set_xlabel('d_min [10^3 km]', color='white')
ax12.tick_params(colors='white'); ax12.grid(True, alpha=0.2, color='white')

# (2,0) Velocidad relativa
ax20 = fig.add_subplot(gs_main[2, 0])
ax20.set_facecolor('#111')
ax20.plot(t_dias, v_rel_km_s, color='#4CAF50', lw=2)
ax20.axvline(t_min_dias, color='#F44336', ls='-.')
ax20.set_title('Vel. relativa Apophis-Tierra', color='white', fontsize=10)
ax20.set_xlabel('Dias', color='white'); ax20.set_ylabel('km/s', color='white')
ax20.tick_params(colors='white'); ax20.grid(True, alpha=0.2, color='white')

# (2,1) Semieje mayor
ax21 = fig.add_subplot(gs_main[2, 1])
ax21.set_facecolor('#111')
ax21.plot(t_elem, a_arr, color='#FDD835', lw=2)
ax21.axvline(t_min_dias, color='#F44336', ls='-.')
ax21.set_title('Semieje mayor heliocentrico', color='white', fontsize=10)
ax21.set_xlabel('Dias', color='white'); ax21.set_ylabel('a [UA]', color='white')
ax21.tick_params(colors='white'); ax21.grid(True, alpha=0.2, color='white')

# (2,2) Excentricidad
ax22 = fig.add_subplot(gs_main[2, 2])
ax22.set_facecolor('#111')
ax22.plot(t_elem, e_arr, color='#E91E63', lw=2)
ax22.axvline(t_min_dias, color='#F44336', ls='-.')
ax22.set_title('Excentricidad heliocentrica', color='white', fontsize=10)
ax22.set_xlabel('Dias', color='white'); ax22.set_ylabel('e', color='white')
ax22.tick_params(colors='white'); ax22.grid(True, alpha=0.2, color='white')

plt.suptitle('PANEL RESUMEN -- 99942 Apophis Flyby 2029\n'
             '5 cuerpos (Sol, Jupiter, Tierra, Luna, Apophis) | IAS15 | SSB',
             color='white', fontsize=14)
plt.savefig('fig10_resumen.png', dpi=150, bbox_inches='tight', facecolor='#0A0A1A')
plt.show()
print('Figura guardada: fig10_resumen.png')



---

## 14. Ejercicios propuestos

### Ejercicio 1 — Integración a largo plazo
Extiende la simulación desde 2020 hasta 2040 (20 años) usando **MERCURIUS** (integrador híbrido de REBOUND). ¿Cuánto cambia la distancia mínima en 2029 respecto a iniciar en 2025?

### Ejercicio 2 — Efecto Yarkovsky
El efecto Yarkovsky produce una aceleración transversal de Apophis de orden a_Y ~ -2.9e-14 AU/dia^2 (Farnocchia et al. 2023). Impleméntalo como fuerza adicional en REBOUND usando `sim.additional_forces`. ¿Cuánto cambia la distancia mínima de 2029?

### Ejercicio 3 — Mapa de probabilidad de impacto
Usando Monte Carlo con N=1000, genera un mapa en el espacio de parámetros iniciales (delta_x, delta_y) coloreado por la distancia mínima. ¿Puedes identificar el *keyhole* (región que llevaría a un impacto en 2036)?

### Ejercicio 4 — Relatividad general
REBOUNDx incluye correcciones relativistas (GR). Compara la distancia mínima con y sin relatividad. ¿Es la diferencia significativa para este encuentro?

### Ejercicio 5 — Animación 3D
Extiende la animación geocéntrica a **3 dimensiones** con `mpl_toolkits.mplot3d`. ¿Se aprecia la inclinación de 3.34° de la órbita de Apophis?


---

## 15. Conclusiones

1. **El encuentro de 2029** fue confirmado como el flyby más cercano jamás predicho para un objeto de más de 100 m: Apophis pasará a ~38 000-40 000 km de la Tierra (~0.1 LD), visible a simple vista.

2. **El baricentro del Sistema Solar (SSB)** es el marco inercial natural para simulaciones de alta precisión. El Sol se desplaza hasta ~1.5 R_sol del SSB, lo que introduce términos de arrastre en el marco heliocéntrico.

3. **Júpiter** modifica la distancia mínima de 2029 en decenas a cientos de km a través de perturbaciones acumuladas en meses. La **Luna** tiene un efecto puntual pero significativo durante el flyby.

4. **La trayectoria geocéntrica** de Apophis es hiperbólica con excentricidad e_h >> 1 (hipérbola muy abierta, flyby rápido). El ángulo de deflexión es de fracciones de grado.

5. **El análisis de Monte Carlo** muestra que dentro de las incertidumbres actuales de posición (+/-50 km), la distancia mínima varía en decenas a cientos de km, lo que no representa ningún riesgo de impacto en 2029.

6. **IAS15** es el integrador correcto para encuentros cercanos. WHFast con paso grande puede cometer errores de miles de km en la distancia mínima.

7. **Los elementos orbitales** de Apophis cambian durante el flyby: el semieje mayor se modifica, lo que altera el período orbital y, en consecuencia, la predicción de encuentros futuros (fenómeno del *keyhole*).

---

## 16. Referencias

[1] Tucker, R., Tholen, D. J., & Bernardi, F. (2004). *IAUC 8477*. IAU.

[2] Giorgini, J. D., et al. (2008). "Predicting the Earth Encounters of (99942) Apophis." *Icarus*, 193, 1-19. https://doi.org/10.1016/j.icarus.2007.09.012

[3] Pravec, P., et al. (2014). "The tumbling spin state of (99942) Apophis." *Icarus*, 233, 48-60.

[4] Rein, H., & Spiegel, D. S. (2015). "IAS15: a fast, adaptive, high-order integrator for gravitational dynamics." *MNRAS*, 446, 1424-1437. https://doi.org/10.1093/mnras/stu2164

[5] Park, R. S., et al. (2021). "The JPL Planetary and Lunar Ephemerides DE441." *AJ*, 161, 105. https://doi.org/10.3847/1538-3881/abd414

[6] Morris, J. (2019). *Celluloid: Simple animation in Matplotlib*. GitHub. https://github.com/jwkvam/celluloid

[7] Farnocchia, D., et al. (2023). "Assessment of the 2068 Impact Probability of (99942) Apophis." *AJ*, 165, 238. https://doi.org/10.3847/1538-3881/acc509

[8] Rein, H., & Liu, S.-F. (2012). "REBOUND: An open-source multi-purpose N-body code." *A&A*, 537, A128. https://doi.org/10.1051/0004-6361/201118085

[9] JPL Horizons Web Interface. https://ssd.jpl.nasa.gov/horizons/

[10] Chesley, S. R., et al. (2003). "Direct Detection of the Yarkovsky Effect by Radar Ranging to Asteroid 6489 Golevka." *Science*, 302, 1739.

[11] de la Fuente Marcos, C., & de la Fuente Marcos, R. (2019). "Dynamical evolution of Apophis co-orbital asteroids." *MNRAS Letters*, 487, L156-L160.

[12] Muller, T. G., et al. (2014). "Thermal infrared observations of asteroid (99942) Apophis." *A&A*, 566, A22.
